In [ ]:
import polars as pl

# Zmień ścieżkę na tę, gdzie trzymasz pobrany plik BindingDB (prawdopodobnie pominięty w .gitignore)
raw_data_path = "datasets/BindingDB_All.tsv"

print("Wczytuję surowy plik...")
try:
    df = pl.read_csv(raw_data_path, separator="	", ignore_errors=False, quote_char=None, infer_schema_length=10000)
    print(f"START (Surowy zbiór): {len(df):,} rekordów
")
except Exception as e:
    print(f"Błąd wczytywania: {e}")
    print("Upewnij się, że plik BindingDB_All.tsv znajduje się w odpowiednim katalogu.")
    import sys; sys.exit(1)

def print_diff(step_name, prev_len, current_len):
    diff = prev_len - current_len
    print(f"{step_name}")
    print(f"Pozostało: {current_len:,} | Odpadło: -{diff:,} rekordów
")
    return current_len

prev = len(df)

# 1. Filtrowanie pustych sekwencji i SMILES (z loader.py)
df = df.filter(pl.col("Ligand SMILES").is_not_null() & pl.col("BindingDB Target Chain Sequence 1").is_not_null())
prev = print_diff("[Krok 1] Usunięcie pustych SMILES i pustych sekwencji białek", prev, len(df))

# 2. Odrzucenie kompleksów wielołańcuchowych (z loader.py)
df = df.with_columns(
    pl.col("Number of Protein Chains in Target (>1 implies a multichain complex)")
    .cast(pl.Int32, strict=False)
    .fill_null(1)
    .alias("n_chains")
)
df = df.filter(pl.col("n_chains") == 1)
prev = print_diff("[Krok 2] Usunięcie kompleksów wielołańcuchowych (n_chains > 1)", prev, len(df))

# 3. Odrzucenie białek z niestandardowymi aminokwasami (z loader.py)
df = df.with_columns(pl.col("BindingDB Target Chain Sequence 1").alias("Full_Protein_Sequence").str.to_uppercase().str.strip_chars())
standard_aa = "ACDEFGHIKLMNPQRSTVWY"
df = df.filter(pl.col("Full_Protein_Sequence").str.contains(f"^[{standard_aa}]+$"))
prev = print_diff("[Krok 3] Usunięcie sekwencji z niestandardowymi aminokwasami", prev, len(df))

# 4. Odrzucenie braku pomiaru Ki (nM) (z loader.py)
# To uderzy we wszystkie wiersze, które miały np. tylko IC50
df = df.with_columns(pl.col("Ki (nM)").str.replace_all(r"[^0-9.]", "").replace("", None).cast(pl.Float64, strict=False))
df = df.filter(pl.col("Ki (nM)").is_not_null() & (pl.col("Ki (nM)") > 0))
prev = print_diff("[Krok 4] Usunięcie rekordów bez poprawnego pomiaru Ki (nM) (np. same IC50/Kd)", prev, len(df))

# 5. Transformacje (usuwanie notacji CX, nulls)
df = df.with_columns(pl.col("Ligand SMILES").str.split_exact("|", 1).struct.field("field_0").str.strip_chars().alias("Ligand SMILES"))
df = df.drop_nulls()
prev = print_diff("[Krok 5] Dodatkowe czyszczenie i drop_nulls() na pozostałych kolumnach", prev, len(df))

# 6. USUNIĘCIE DUPLIKATÓW (Średnia pomiarów dla tej samej pary) (z transform.py)
df = df.group_by(["Ligand SMILES", "Full_Protein_Sequence"]).agg([pl.col("Ki (nM)").mean().alias("Ki (nM)")])
prev = print_diff("[Krok 6] Agregacja duplikatów (średnia Ki) dla tych samych par Lek-Białko", prev, len(df))

# 7. Usunięcie niepoprawnych SMILES przez RDKit (z transform.py)
from rdkit import Chem
def is_valid(smi: str) -> bool:
    try:
        return Chem.MolFromSmiles(smi) is not None
    except Exception:
        return False

print("Walidacja RDKitem... (to potrwa chwilę)")
valid_mask = df["Ligand SMILES"].map_elements(is_valid, return_dtype=pl.Boolean)
df = df.filter(valid_mask)
prev = print_diff("[Krok 7] Usunięcie niepoprawnych chemicznie SMILES (RDKit)", prev, len(df))

print(f"ZAKOŃCZONO. Ostateczny zbiór danych (powinno być koło 450k): {len(df):,}")
